In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists


RUN_API_AUTH_TEST = True
HEADER_FILE = '../headers_auth.json'
PLAYLIST_TSV_DIR = '../playlists/'


In [8]:
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")


Using header file: ../headers_auth.json
Test Passed in 2.59 seconds
Using ytmusicapi version: 0.25.0
Loaded 482 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [13]:
playlist_names = [
    # 'x_r.WorldMusic_tracks_radio', # waaking caos
    # 'Soul Classic Sunshine radio',
    # 'rock instrumentals classic vintage radio', # moonchild ventures
    # 'jazz guitar radio', # golden earing
    # 'rock instrumentals classic vintage radio',
    # 'Soul Classic Sunshine radio'
    # 'electronic House Special radio', # show me
    # 'psychedelic classic rock radio',
    # 'electronic soft pad radio', # hatchet
    # 'Indie Dreams of Fall radio'
    # 'Reggae Dub radio' # natty dread ubn
    # 'electronic soft pad radio'
    # 'Hiphop southeast Ride Around Shining radio'
    
]
MIN_RADIO_LIKE_TO_SPLIT=2
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_playlist(pl_info, 
        move_like=True, create_like_playlist=True, remove_dislike=True, remove_not_like=True,
        min_num_like=MIN_RADIO_LIKE_TO_SPLIT,  sleep=1, verbose=True,
    )


****************************************************************************************************
{'name': 'Hiphop southeast Ride Around Shining radio', 'removed_dislike': 0, 'moved_like': 3, 'removed_not_like': 0, 'like_and_not_like': 0}
Moved 3 LIKE entries from Hiphop southeast Ride Around Shining radio to PLWptjpDqazOwbWvd79avLwIkyZ-Czt7BU


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [3]:
playlist_names = [
    'jazz gloom smooth radio'
]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=USE_CACHE)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


Loaded 266580 playounts from 104345 tracks
Created sorted pl: PLWptjpDqazOzoAgaNRdpvi4O3jK3lG0AC, and  deleted original pl: PLWptjpDqazOxD9ZM736lGLmWw5BQB-Z5D


## Query

In [ ]:
Y.query_by_title(playlist_name)

## Save playlist backup tsv

In [ ]:
PLAYLIST_NAME = 'zz not like 2'
USE_CACHE = False
_pl_id = Y.query_by_title(PLAYLIST_NAME).playlistId
_pl_tracks, _pl_metadata = Y.save_playlist_tsv(Y.playlist_get_info(_pl_id, use_cache=USE_CACHE))

## Upload playlist from tsv backup

In [11]:
PLAYLIST_NAME = 'x_r.FunkSouMusic_tracks_like'
playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
Y.playlist_from_tsv(playlist_file, ignore_banned=True, sort_by_index=True)


Generating x_r.FunkSouMusic_tracks_like ytmusic playlist for 163 tracks
Saved 163 x_r.FunkSouMusic_tracks_like tracks playlist with id: PLWptjpDqazOzJIuIhuHxrw3iKLisUhVAJ


## Get Playlist Counts

In [6]:
%%time
playlist_file = os.path.join(PLAYLIST_TSV_DIR, '_playlist_radio_counts.tsv')
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
radio_counts_df = playlists.loc[playlists.title.str.contains('radio')].sort_values('track_count')
radio_counts_df = radio_counts_df[['title', 'track_count', 'duration_hours', 'privacy', 'playlist_id']]
radio_counts_df.to_csv(playlist_file, sep='\t', index=False)
radio_counts_df.head(20)
# last run 7-17-2023


CPU times: total: 32.9 s
Wall time: 4min 2s


,title,track_count,duration_hours,privacy,playlist_id
135,Hiphop southeast Ride Around Shining radio,17,1,PRIVATE,PLWptjpDqazOy0aKWVQk39xpseuXZ7s5AL
25,rock 1967 Monterey Pop Festival radio,33,2,PRIVATE,PLWptjpDqazOyoXjkoYrtOq-HxIFikPMb0
137,jazz essential radio,35,4,PRIVATE,PLWptjpDqazOxDKPeCgfbu_jp3Edn-Sh8M
0,jazz gloom smooth radio,38,4,PRIVATE,PLWptjpDqazOzoAgaNRdpvi4O3jK3lG0AC
121,Indie dark side radio,38,3,PRIVATE,PLWptjpDqazOy4SxjT3JAAOtH2cpLu454G
154,x_r.MusicToSleepTo_tracks_radio,40,6,PRIVATE,PLWptjpDqazOzfumkZJ8sZi_fKVxXLZOfI
142,jazz traditional radio,41,3,PRIVATE,PLWptjpDqazOxgPp1RGviOEtXoUSwxcsJo
64,electronic soft pad radio,43,3,PRIVATE,PLWptjpDqazOxVocLsdDXRaXoLUaUDPSak
153,x_r.chicagohouse_tracks_radio,47,5,PRIVATE,PLWptjpDqazOzwOd5TBZvCFqkwH5hcuMbu
133,Hip hop It Was a Good Day radio,47,3,PRIVATE,PLWptjpDqazOz-tIqgqECFiRHLqE5IYtdz


## Update _not_like tsv

In [9]:
not_like_tracks = Y.collect_all_not_like_tracks_from_tsvs()
not_like_tracks.to_csv(Y.not_like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to not_like tsv: {Y.not_like_tsv}')

Found 2 not like playlists out ot the 498 total
Updated not liked tracks, contains 4996 entries.
Saved 4996 entries to not_like tsv: ../playlists/_not_liked_tracks.tsv


## Update _like tsv and print like coverage

In [10]:
like_tracks = Y.collect_all_like_tracks_from_tsvs()
like_tracks.to_csv(Y.like_tsv, sep='\t', header=True)
print(f'Saved {len(not_like_tracks)} entries to like tsv: {Y.like_tsv}')

Found 114 like playlists out ot the 498 total
Beats indie Chill.tsv	100.0% currently liked (of 83 total tracks) 
Bossa Nova.tsv	95.2% currently liked (of 21 total tracks) 
Brass n chill.tsv	99.0% currently liked (of 100 total tracks) 
Chillwave.tsv	100.0% currently liked (of 100 total tracks) 
Folk.tsv	100.0% currently liked (of 249 total tracks) 
Grunge.tsv	100.0% currently liked (of 79 total tracks) 
Hip Hop 1990s.tsv	98.9% currently liked (of 620 total tracks) 
Hip Hop 2000s.tsv	100.0% currently liked (of 92 total tracks) 
Hip Hop Hits.tsv	99.1% currently liked (of 218 total tracks) 
Hip hop It Was a Good Day.tsv	98.7% currently liked (of 77 total tracks) 
Indie 1990s Rock.tsv	100.0% currently liked (of 76 total tracks) 
Indie 2000s.tsv	99.5% currently liked (of 197 total tracks) 
Jazz Guitar.tsv	98.4% currently liked (of 61 total tracks) 
Oldies.tsv	99.5% currently liked (of 623 total tracks) 
Post-Punk 1970s-1980s.tsv	100.0% currently liked (of 88 total tracks) 
Produced By Dilla.

## Get Public Playlists

In [7]:
Y.get_playlists_by_privacy(privacy='PUBLIC')
# last run 6-2023, 3 public playlists

Found public playlist named: Liked Music
Found public playlist named: Chill Supermix
Found public playlist named: Episodes for Later


title                                                Liked Music
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
author                                                       NaN
title                                             Chill Supermix
playlistId           RDTMAK5uy_nzfwl2UYv7htL7wDoxbX8Pp6UAFBd92cQ
thumbnails     [{'url': 'https://music.youtube.com/image/mixa...
description                                        YouTube Music
count                                                        NaN
author                                                       NaN
title                                         Episodes for Later
playlistId                                                    SE
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description              

### Generate playlist froma a list of albums

In [ ]:
name = 'y_2022_albums_to_listen_to'
desc = 'manually selected albums to top off 2022 albums'
albums_to_add = [
    "SZA - SOS",
    "Perfume Genius - Ugly Season",
    "Arctic Monkeys - The Car",
    "The Beths - Expert in a Dying Field",
    "alt-J - The Dream",
     "Soichi Terada (寺田創一) - Asakusa Light",
    "Everything Everything - Raw Data Feel",
    "The Mountain Goats - Bleed Out",
    "Metric - Formentera",
    "Orville Peck - Bronco",
    "Hurray for the Riff Raff - Life on Earth",
    "Freddie Gibbs - $oul $old $eparately",
    "Tove Lo - Dirt Femme",
    "Oren Ambarchi - Shebang",
    "Hatchie - Giving the World Away",
    "Florence + the Machine - Dance Fever",
    "Richard Dawson - The Ruby Cord",
    "Charlotte Adigéry & Bolis Pupul - Topical Dancer",
    "Phoenix (FR) - Alpha Zulu",
    "Elder - Innate Passage",
    "The Black Angels - Wilderness of Mirrors",
    "Black Flower - Magma",
     "Bad Bunny - Un Verano Sin Ti",
     "Red Hot Chili Peppers - Return Of The Dream Canteen",

]

track_ids = []
for a in albums_to_add:
    match = {}
    res = Y.yt.search(query=a, filter='albums', limit=1)
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    if len(res) == 0 or res[0].get('browseId') == None:
        print(f'Skipping query: {a} bad result: {res}')
        continue
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    print(f'Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album}')

pl_id = Y.yt.create_playlist( title=name,  description=desc, video_ids=track_ids)
print(f'Generated playlist: {name} with id {pl_id}')